In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize']= (12,6)

In [ ]:
df= pd.read_csv('../../data/train_optimized.csv')

In [ ]:
df.head()

## Kategorik ve Numerik Değişkenlerin Ayrılması

In [ ]:
# 1. Object ve Category tipindeki kolonlar (gerçek kategorikler)
true_categorical = df.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Numerik tipte ama benzersiz değer sayısı düşük olanlar
numeric_cols = df.select_dtypes(include=['int64', 'float64', 'Int8', 'Int16', 'Int32', 'Int64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['TransactionID', 'isFraud']]

numeric_but_categorical = []
for col in numeric_cols:
    if df[col].nunique() <= 20:
        numeric_but_categorical.append(col)

# 3. Tüm kategorik değişkenler
categorical_vars = true_categorical + numeric_but_categorical

# 4. Gerçek numerik değişkenler
numeric_vars = [col for col in numeric_cols if col not in numeric_but_categorical]

print(f"Kategorik değişkenler: {len(categorical_vars)}")
print(f"Numerik değişkenler: {len(numeric_vars)}")

In [ ]:
# Kategorik değişkenleri cardinality'ye göre grupla
categorical_info = []
for col in categorical_vars:
    categorical_info.append({
        'Kolon': col,
        'Tip': str(df[col].dtype),
        'Benzersiz Değer': df[col].nunique(),
        'Eksik %': f"{(df[col].isnull().sum() / len(df)) * 100:.2f}%"
    })

cat_df = pd.DataFrame(categorical_info).sort_values('Benzersiz Değer', ascending=False)

print("Yüksek Cardinality (>50):")
print(cat_df[cat_df['Benzersiz Değer'] > 50])

print("\nOrta Cardinality (11-50):")
print(cat_df[(cat_df['Benzersiz Değer'] > 10) & (cat_df['Benzersiz Değer'] <= 50)])

print("\nDüşük Cardinality (≤10):")
print(cat_df[cat_df['Benzersiz Değer'] <= 10])

In [ ]:
# Visualize distribution
fig, ax = plt.subplots(figsize=(10, 6))

cardinality_groups = {
    'Düşük (≤10)': len(cat_df[cat_df['Benzersiz Değer'] <= 10]),
    'Orta (11-50)': len(cat_df[(cat_df['Benzersiz Değer'] > 10) & (cat_df['Benzersiz Değer'] <= 50)]),
    'Yüksek (>50)': len(cat_df[cat_df['Benzersiz Değer'] > 50])
}

ax.bar(cardinality_groups.keys(), cardinality_groups.values())
ax.set_title('Kategorik Değişkenlerin Cardinality Dağılımı')
ax.set_ylabel('Değişken Sayısı')
plt.show()